# Guided tour of the financials factor pipeline

An end-to-end walk from **raw data → panel → factors → validation & selection**, using
the real modules under `src/`. The emphasis is the final stage: turning ~20 raw style
factors into a **shortlisted, non-redundant set** via Information Coefficient, Fama-MacBeth,
redundancy clustering, and a lasso.

The strategy universe is **financials (banks + insurers)**. Banks and insurers carry
mutually-exclusive metrics, so the pipeline is *sub-universe aware* throughout: factors are
partitioned by `Applicability`, and the cross-sectional steps never mix the two sleeves.

> **Scope knobs.** The full panel is ~150M price rows. To keep the tour to well under a
> minute we restrict to a multi-region set of countries and a 10-year window (set below).
> Widen `COUNTRIES` / `WINDOW` to run larger slices — the code is identical.

In [ ]:
import sys, functools, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import polars as pl
pl.enable_string_cache()   # stock_id is Categorical; a shared cache makes joins reliable
from plotnine import (ggplot, aes, geom_col, geom_line, geom_point, geom_hline,
                      coord_flip, facet_wrap, labs, theme_bw, theme, element_text)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config, universe
from src.data import loaders, joins, preprocess
from src.factors.base import registry
from src.validation import single_factor as sf, redundancy as rd, neutralization as nz

cfg = config.load(PROJECT_ROOT / "config.yaml")
cfg.raw["data"]["root"] = str(PROJECT_ROOT / cfg["data"]["root"])   # anchor relative paths

# ── tour knobs ────────────────────────────────────────────────────────────────
COUNTRIES = ["US", "JP", "GB", "FR", "DE", "HK", "AU", "KR"]
WINDOW = ("2013-01-01", "2022-12-31")
W = (pl.lit(WINDOW[0]).str.to_date(), pl.lit(WINDOW[1]).str.to_date())
pl.Config.set_tbl_rows(30)

## 1. Ingestion

`loaders.load_all` returns every raw table as a **lazy** `scan_ipc` frame — nothing is read
until a `.collect()`. That lets us prune to the universe/window before materialising anything.

In [ ]:
raw = loaders.load_all(cfg)
print("tables:", sorted(raw))
raw["price"].collect_schema()   # daily price / return / mcap

## 2. Point-in-time panel & universe

`joins.build_panel` performs the look-ahead-safe assembly (as-of fundamentals, FX to USD,
excess return, Nelson-Siegel curve params). We then:

* `preprocess.clean` — repair returns, drop rows with no risk-free rate, recompute excess return;
* `universe.sector_set` — restrict to banks + insurance (GICS ∩ FactSet ∩ populated sector metric);
* `universe.industry_labels` — attach the `industry` sub-universe label (`bank` / `insurance_*`).

We pre-filter the heavy `price` / `zero_curve` tables to the tour's ids, window and currencies
first, so `build_panel` only touches the slice we need.

In [ ]:
sm = raw["security_master"]
ids = (sm.filter(pl.col("gics_industry_name").is_in(["Banks", "Insurance"]))
         .filter(pl.col("country_code").is_in(COUNTRIES))
         .select("stock_id").unique().collect())
ccys = (sm.join(ids.lazy(), on="stock_id", how="inner")
          .select("currency_code").unique().collect())["currency_code"].to_list()

raw["price"] = (raw["price"].filter(pl.col("date").is_between(*W))
                .join(ids.lazy(), on="stock_id", how="inner"))
raw["zero_curve"] = (raw["zero_curve"].filter(pl.col("date").is_between(*W))
                     .filter(pl.col("currency").is_in(ccys)))

panel = joins.build_panel(raw, cfg).collect()
panel = universe.industry_labels(universe.sector_set(preprocess.clean(panel, cfg), cfg), cfg)

print(f"panel: {panel.height:,} rows x {panel.width} cols, "
      f"{panel['stock_id'].n_unique()} stocks, {panel['date'].min()} .. {panel['date'].max()}")
panel.group_by("industry").len().sort("len", descending=True)

## 3. Factor generation

The registry enumerates every factor; each `compute` returns a uniform
`[stock_id, date, <shorthand>]` frame. **Style** factors are cross-sectional ratios;
**non-style** structural factors (Market / Country / Industry) are per-security *loadings*
— rolling time-series betas on their factor-mimicking-portfolio returns.

We seed every wide join with the full `(stock_id, date)` grid so the sparse quarterly
yield-curve factors can't shrink the row set.

In [ ]:
reg = registry()
STYLE    = {s: c for s, c in reg.items() if c.__module__.startswith("src.factors.style")}
NONSTYLE = {s: c for s, c in reg.items() if c.__module__.startswith("src.factors.nonstyle")}
SLEEVE   = {s: c.sleeve for s, c in reg.items()}
print("style   :", sorted(STYLE))
print("nonstyle:", sorted(NONSTYLE))

def join_all(frames):
    return functools.reduce(lambda a, b: a.join(b, on=["stock_id", "date"], how="left"), frames)

grid = panel.select("stock_id", "date")
scores  = join_all([panel.select("stock_id", "date", "industry")]
                   + [STYLE[s]().compute(panel, cfg) for s in STYLE])
allload = join_all([grid] + [NONSTYLE[s]().compute(panel, cfg) for s in NONSTYLE])
print("style scores:", scores.shape, "| non-style loadings:", allload.shape)

Not all non-style loadings are usable as neutralization regressors. The yield-curve
sensitivities are **quarterly and sub-universe-specific**, so they are populated on only a
sliver of rows — feeding them into one pooled cross-sectional regression (which drops any row
with a null regressor) would wipe out the panel. We neutralize against the **dense structural
trio** `MKT / CTRY / IND`.

In [ ]:
lcols = [c for c in allload.columns if c not in ("stock_id", "date")]
coverage = (pl.DataFrame({"loading": lcols})
            .with_columns(pct_populated=pl.Series(
                [round(allload[c].drop_nulls().len() / allload.height, 3) for c in lcols]))
            .sort("pct_populated", descending=True))
STRUCTURAL = ["MKT", "CTRY", "IND"]
loadings = allload.select("stock_id", "date", *STRUCTURAL)
coverage

## 4. Regularize & neutralize

`preprocess.regularize` winsorizes → median-imputes → z-scores **within each factor's
sub-universe** (a bank never receives an insurer's median, and jointly-computable factors like
`B/P` are nulled outside their sub-universe). `neutralization.neutralize` then replaces each
style score with its residual from a per-date cross-sectional OLS on the structural loadings —
the part of the exposure *not* explained by market/country/industry risk.

In [ ]:
regd = preprocess.regularize(scores, cfg)
neu  = nz.neutralize(regd, loadings)

# sub-universe partitioning in action: B/P is an insurance factor (computable for banks too),
# so it is present for insurers and null for banks after regularisation.
bp_bank = neu.filter(pl.col("industry") == "bank")["B/P"].drop_nulls().len()
bp_ins  = neu.filter(pl.col("industry").str.starts_with("insurance"))["B/P"].drop_nulls().len()
print(f"neutralized {neu.shape}; B/P non-null  banks={bp_bank}  insurers={bp_ins}")
print("(loadings warm up over ~2y, so early residuals are null and drop out of validation)")

## 5. Validation & selection

This is the heart of the tour. Four complementary lenses, then a combined scorecard.

### 5a. Information Coefficient & IC decay
Rank IC = per-period Spearman correlation between the neutralized exposure and the forward
quarterly return. Its mean/std is the **IC-IR**; a factor is shortlisted at `|IR| > 0.3`.
Horizons `1,2,3` (quarters) trace the **decay** curve.

In [ ]:
ic = sf.rank_ic(neu, panel.select("stock_id", "date", "excess_return"),
                lags=tuple(cfg["validation"]["ic"]["decay_lags"]))
ir = sf.information_ratio(ic)
THR = cfg["validation"]["ic"]["ir_shortlist_threshold"]

ir1 = ir.filter(pl.col("lag") == 1).drop_nulls("ir").sort(pl.col("ir").abs(), descending=True)
shortlist_ir = set(ir1.filter(pl.col("ir").abs() > THR)["factor"])
print(f"IR shortlist (|IR| > {THR}):", sorted(shortlist_ir))
ir1.head(10)

In [ ]:
ir1p = ir1.with_columns(shortlisted=pl.col("ir").abs() > THR).to_pandas()
(ggplot(ir1p, aes("reorder(factor, ir)", "ir", fill="shortlisted"))
 + geom_col() + coord_flip()
 + geom_hline(yintercept=[-THR, THR], linetype="dashed")
 + labs(title="IC-IR by factor (lag 1 = one quarter ahead)", x="", y="IC-IR")
 + theme_bw())

In [ ]:
decay = (ic.filter(pl.col("factor").is_in(sorted(shortlist_ir)))
         .group_by("factor", "lag").agg(ic=pl.col("ic").mean()).sort("factor", "lag").to_pandas())
(ggplot(decay, aes("lag", "ic", color="factor"))
 + geom_line() + geom_point()
 + labs(title="IC decay of shortlisted factors", x="horizon (quarters)", y="mean IC")
 + theme_bw())

### 5b. Fama-MacBeth premia
Per-period cross-sectional regressions of forward returns on the factors, **run separately per
sub-universe** (banks and insurers hold disjoint factors), aggregated with Newey-West t-stats.

In [ ]:
fm = sf.fama_macbeth(neu, panel.select("stock_id", "date", "excess_return")).filter(pl.col("factor") != "const")
fm_sig = set(fm.filter(pl.col("t_stat").abs() > 2)["factor"])
print("FM significant (|t| > 2):", sorted(fm_sig))
fm.sort(pl.col("t_stat").abs(), descending=True, nulls_last=True).head(10)

In [ ]:
fmp = fm.drop_nulls("t_stat").to_pandas()
(ggplot(fmp, aes("reorder(factor, t_stat)", "t_stat"))
 + geom_col() + coord_flip() + facet_wrap("sub_universe", scales="free_y")
 + geom_hline(yintercept=[-2, 2], linetype="dashed")
 + labs(title="Fama-MacBeth premia (Newey-West t)", x="", y="t-stat")
 + theme_bw())

### 5c. Redundancy — correlation clustering
Flag factor pairs whose time-averaged cross-sectional correlation exceeds the threshold, group
them into clusters (connected components), and keep the **highest-IR representative** of each
cluster (dropping the rest as redundant).

In [ ]:
pairs, flagged = rd.average_correlation(neu, threshold=cfg["validation"]["redundancy"]["correlation_threshold"])
clusters = rd.cluster_factors((pairs, flagged))
reps = set(rd.select_cluster_representatives(clusters, ic_ir=ir))
print("flagged pairs:", flagged.height)
print("correlated clusters:", [c for c in clusters if len(c) > 1])
print("kept representatives:", len(reps), "of", len(STYLE), "factors")
flagged

### 5d. Parsimony — lasso
A cross-validated lasso predictive regression (per sub-universe, **CV folds grouped by period**
to avoid look-ahead leakage). Survivors are the non-shrunk factors.

In [ ]:
lasso = set(rd.lasso_select(panel.select("stock_id", "date", "excess_return"), neu, cfg))
print("lasso survivors:", sorted(lasso))

### 5e. Final selection scorecard
Combine the lenses: keep a factor when it is a **cluster representative** (non-redundant) *and*
has predictive evidence from **at least one** of IR / Fama-MacBeth / lasso.

In [ ]:
ir_best = ir.group_by("factor").agg(ic_ir=pl.col("ir").max()).select("factor", "ic_ir")
fm_best = (fm.sort(pl.col("t_stat").abs(), descending=True, nulls_last=True)
           .unique("factor", keep="first").select("factor", fm_t="t_stat"))

scorecard = (pl.DataFrame({"factor": sorted(STYLE)})
    .join(ir_best, on="factor", how="left")
    .join(fm_best, on="factor", how="left")
    .with_columns(
        sleeve=pl.col("factor").replace_strict(SLEEVE, default=None),
        representative=pl.col("factor").is_in(list(reps)),
        ir_pass=pl.col("factor").is_in(list(shortlist_ir)),
        fm_sig=pl.col("factor").is_in(list(fm_sig)),
        lasso=pl.col("factor").is_in(list(lasso)))
    .with_columns(keep=pl.col("representative") & (pl.col("ir_pass") | pl.col("fm_sig") | pl.col("lasso")))
    .sort("keep", pl.col("ic_ir").abs(), descending=True, nulls_last=True))

final = sorted(scorecard.filter(pl.col("keep"))["factor"])
print("FINAL SHORTLIST:", final)
scorecard

## Caveats & scaling up

* **Working slice.** Results above are for the `COUNTRIES` / `WINDOW` slice; widen them (or drop
  the pre-filter) to run the full universe — runtime scales roughly with row count.
* **Loadings warm-up.** Structural betas need ~2 years of history, so the first window years and
  the newest listings carry null loadings and drop out of neutralization/validation.
* **Yield-curve factors.** Kept out of the pooled neutralization because they are quarterly and
  sub-universe-specific (see the coverage table); a per-sub-universe neutralization would let
  them back in.
* **Currency factor** is not yet implemented (`src/factors/nonstyle/currency.py`).
* The IC/IR thresholds, correlation cut-off, rebalancing frequency and Newey-West lags are all
  read from `config.yaml`.